# MagicVision YOLO Training - Google Colab
Entrenamiento y validacion fuera de AWS; empaquetado del checkpoint completo para inferencia en SageMaker.

In [ ]:
!pip install -q ultralytics opencv-python roboflow kaggle
from ultralytics import YOLO
import os, shutil, pathlib
print('Entorno listo')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = pathlib.Path('/content/drive/MyDrive/MagicVision')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_DIR)

## Dataset

Sube o descarga un dataset YOLO con esta estructura:

`dataset_final_magicvision/images/train`, `images/val`, `images/test`, `labels/train`, `labels/val`, `labels/test`.

Puedes ejecutar los scripts de `ml/scripts` desde tu repo sincronizado en Drive.

In [ ]:
DATASET_DIR = PROJECT_DIR / 'ml/datasets/dataset_final_magicvision'
DATA_YAML = PROJECT_DIR / 'ml/configs/data.yaml'
print('Dataset:', DATASET_DIR)
print('data.yaml:', DATA_YAML)
assert DATA_YAML.exists(), 'Copia ml/configs/data.yaml a Drive o ajusta DATA_YAML'

In [ ]:
# Validacion rapida de estructura
for split in ['train', 'val', 'test']:
    print(split, len(list((DATASET_DIR / 'images' / split).glob('*'))), 'imagenes')
    print(split, len(list((DATASET_DIR / 'labels' / split).glob('*.txt'))), 'labels')

In [ ]:
# Entrenar YOLO nano para movil
model = YOLO('yolo11n.pt')
results = model.train(data=str(DATA_YAML), epochs=80, imgsz=640, batch=16, project=str(PROJECT_DIR / 'runs/detect'), name='magicvision_yolo')

In [ ]:
# Evaluar mejor peso
best = PROJECT_DIR / 'runs/detect/magicvision_yolo/weights/best.pt'
model = YOLO(str(best))
metrics = model.val(data=str(DATA_YAML), imgsz=640)
print(metrics)

In [ ]:
# Empaquetar pesos completos. No conversion ni inferencia en Android.
import subprocess
subprocess.run(['python', str(PROJECT_DIR / 'backend/package_model.py'), '--weights', str(best), '--labels', str(PROJECT_DIR / 'ml/configs/classes.txt'), '--output', str(PROJECT_DIR / 'backend/artifacts/model.tar.gz')], check=True)


In [ ]:
MODEL_ARTIFACT = PROJECT_DIR / 'backend/artifacts/model.tar.gz'
print('Artefacto para S3/SageMaker:', MODEL_ARTIFACT)


In [ ]:
from google.colab import files
files.download(str(MODEL_ARTIFACT))
